#### ChromaDB Learning

[Documentation Link](https://docs.trychroma.com/docs/collections/manage-collections)

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# %%capture
# ! pip install chromadb

##### 1. Initializing Directories and importing Libs

In [3]:
import os

In [4]:
ingestion_doc_path = os.path.join(os.getcwd(), 'ingestion_data')
chroma_db_persist_dir = os.path.join(os.getcwd(), 'chroma_store')

In [5]:
import chromadb

In [6]:
# creating chromadb client
# chroma_client = chromadb.Client() # In Memory Client
chroma_persistent_client = chromadb.PersistentClient(path=chroma_db_persist_dir)

##### 2. ChromaDB Setup

ChromaDB Collections:

Collections are where we'll store our embeddings, documents, and any additional metadata. Collections index our embeddings and documents, and enable efficient retrieval and filtering. You can create a collection with a name:

In [7]:
# creating collection
collection_name = "heros_collection"

if collection_name in [item.name for item in chroma_persistent_client.list_collections()]:
  chroma_persistent_client.delete_collection(name=collection_name)

heros_collection = chroma_persistent_client.create_collection(name=collection_name)

print("document count:", heros_collection.count())


document count: 0


Chroma Collection - add: Add method adds a new required despite of being duplicate

Chroma Collection - upsert: Upsert method update in case exists otherwise inserts.

Let's first get the document and prepare the metadata for them

In [8]:
import os
import random

##### 3. Data Prep and Feed

In [15]:
all_txt_files = os.listdir(ingestion_doc_path)

documents_details_list = []
counter = 0
for file_name in all_txt_files:
  counter+=1
  document_details_dict = {}
  document_details_dict["id"] = "doc_" + str(counter)
  with open(os.path.join(ingestion_doc_path, file_name)) as tFile:
    document_details_dict["content"] = tFile.read()

  document_details_dict["metadata"] = {
      "doc_id": document_details_dict["id"],
      "doc_name": file_name.split(".")[0],
      "doc_full_name": file_name,
      "doc_content_length": len(document_details_dict["content"]),
      "superstar_rating": random.randint(6,10)
  }

  # now adding these into main list:
  documents_details_list.append(document_details_dict)

print("length of document list: ", len(documents_details_list) )

length of document list:  19


Now, let's split and create different list as per chroma need

In [16]:
# before upserting let's split the info for document

documents =  [item["content"] for item in documents_details_list]
ids =  [item["id"] for item in documents_details_list]
metadatas =  [item["metadata"] for item in documents_details_list]

let's add the data into collection

In [17]:
# let's add into collection, it will create the embeddings. if we give "embeddings" paramter then it will skip creating embeddings.

heros_collection.add(
    documents = documents,
    ids = ids,
    metadatas = metadatas
)

print("document count", heros_collection.count())

document count 19


##### 4. Query the ChromaDB

Perfect! let's query with simple text

In [18]:
result = heros_collection.query(
    query_texts = ["who holds Mjolnir"],
    n_results = 2
)
result

{'ids': [['doc_17', 'doc_12']],
 'embeddings': None,
 'documents': [['The God of Thunder: Thor Odinson, Heir of Asgard\nThe legend begins in the celestial realm of Asgard, one of the Nine Realms, where Thor Odinson was born the crown prince, son of Odin Allfather, the ruler of Asgard, and Gaea, the Elder Goddess of the Earth. From his earliest days, Thor was characterized by boundless strength, fiery courage, and, often, a crippling arrogance. He was trained relentlessly in the arts of combat, becoming the mightiest warrior in Asgard. His defining symbol and tool is Mjolnir, the enchanted uru hammer, forged in the heart of a dying star and imbued with Odin\'s powerful enchantment: "Whosoever holds this hammer, if he be worthy, shall possess the power of Thor." It was this enchantment that would define his entire existence. Due to his growing impulsiveness and hubris, which threatened to plunge Asgard into war, Odin eventually banished Thor to Earth (Midgard), stripped of his memory and

let's look into metadata

In [19]:
result["metadatas"]

[[{'doc_content_length': 6146,
   'doc_full_name': 'thor.txt',
   'doc_id': 'doc_17',
   'doc_name': 'thor',
   'superstar_rating': 6},
  {'doc_content_length': 5756,
   'doc_full_name': 'odin.txt',
   'doc_name': 'odin',
   'doc_id': 'doc_12',
   'superstar_rating': 7}]]

Now, let's query with meatadata filter, so we have to use 'where' clause

In [21]:
result = heros_collection.query(
    query_texts = ["who holds Mjolnir"],
    n_results = 1,
    where={"doc_name":"odin"}
)
result

{'ids': [['doc_12']],
 'embeddings': None,
 'documents': [['The Allfather: Odin Borson, King of Asgard and Lord of the Nine Realms\nThe immense legend of Odin begins in the primordial past, as the son of Bor, and the true architect of the current iteration of the cosmos. Known officially as Odin Borson, he is the supreme ruler of Asgard, and the self-proclaimed benevolent, yet iron-fisted, sovereign of the Nine Realms. His life is measured not in decades, but in epochs, having ruled Asgard and fiercely defended the cosmos against threats far older than Earth\'s history. His early years were defined by colossal, realm-spanning warfare, during which he and his brothers defeated ancient enemies, including the fire giant Surtur, and established the rigid, hierarchical order of the Nine Realms. The discovery of the infant Frost Giant, Loki, and his subsequent adoption was a defining act—a complex decision that was meant to forge peace but instead created the central, generational conflict o

nice, likewise we can apply many filters upon available metadata

Getting results with only metadata

In [28]:
result = heros_collection.get( 
    where={"doc_name":"odin"}
)
result

{'ids': ['doc_12'],
 'embeddings': None,
 'documents': ['The Allfather: Odin Borson, King of Asgard and Lord of the Nine Realms\nThe immense legend of Odin begins in the primordial past, as the son of Bor, and the true architect of the current iteration of the cosmos. Known officially as Odin Borson, he is the supreme ruler of Asgard, and the self-proclaimed benevolent, yet iron-fisted, sovereign of the Nine Realms. His life is measured not in decades, but in epochs, having ruled Asgard and fiercely defended the cosmos against threats far older than Earth\'s history. His early years were defined by colossal, realm-spanning warfare, during which he and his brothers defeated ancient enemies, including the fire giant Surtur, and established the rigid, hierarchical order of the Nine Realms. The discovery of the infant Frost Giant, Loki, and his subsequent adoption was a defining act—a complex decision that was meant to forge peace but instead created the central, generational conflict of h

##### 4. Update the records

In [22]:
# let's update the odin rating, 'doc_id': 'doc_12'
# letn's update the rating
heros_collection.update(
    ids=['doc_12'],  # odin's doc id
    metadatas=[{"superstar_rating":100}]
)
print("document updated")

document updated


Perfect! Let's ask again and look into only metadata

In [ ]:
# to use only metadata filter, we have to use get method instead query
# we can also set limit, offset and rest of the things
heros_collection.get(
     where={"doc_id":"doc_12"}
)["metadatas"]

[{'doc_name': 'odin',
  'doc_id': 'doc_12',
  'doc_content_length': 5756,
  'doc_full_name': 'odin.txt',
  'superstar_rating': 100}]

Bingooo!!!! Could see superstar rating updated!!!

#### Trying Some other operations

In [ ]:
# adding another record into collection

heros_collection.add(
    ids=['doc404'],
    documents=['This is error document'],
    metadatas=[{
        "doc_id":"doc404",
        "doc_name": 'error',
        "doc_full_name": 'error.txt',
        "doc_content_length": 30,
        "superstar_rating": 0
    }]
)

print("document added into collection:- ", heros_collection.count())

document added into collection:-  20


let's update the document content

In [ ]:
new_content = "The error has been addressed. Now this could be the rising hero."

heros_collection.update(
    ids=['doc404'],
    documents=[new_content],
    metadatas=[{'doc_content_length':80}]
)

print("document has updated")

document has updated


Let's view the updated document

In [ ]:
heros_collection.get('doc404')

{'ids': ['doc404'],
 'embeddings': None,
 'documents': ['The error has been addressed. Now this could be the rising hero.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'doc_full_name': 'error.txt',
   'doc_name': 'error',
   'doc_id': 'doc404',
   'superstar_rating': 0,
   'doc_content_length': 80}]}

Nice, its updated